# ST-OMR Meter V5-3H — Authoritative Rescue TRAIN Execution\n\nFail-closed exact-SHA wrapper. It uses an isolated Python venv so Colab preinstalled packages cannot alter the pinned V5-3G rescue runtime. Protected evaluation gates remain closed.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, shutil, subprocess, sys, textwrap

EXPECTED_HEAD = "b36a9d2f5daade2c3568cac8cbc736ca75ca435f"
EXPECTED_CI_RUN_ID = 32769348282
REPOSITORY = "khfy7wpr5p-maker/st-omr-training"
REPO = Path("/content/st-omr-training")
MYDRIVE = Path("/content/drive/MyDrive")
VENV = Path("/content/st-omr-v5-3h-venv")
VENV_PYTHON = VENV / "bin" / "python"

if not MYDRIVE.is_dir():
    from google.colab import drive
    drive.mount("/content/drive")

DATA_ROOT = MYDRIVE / "TEST" / "METER_V2_1500_PACKAGE_AB_CLEAN"
CHECKPOINT_ROOT = MYDRIVE / "ST-OMR-METER-SPECIALISTS"
M4A_ROOT = CHECKPOINT_ROOT / "m4a-234-digit-specialist-dataset-freeze-v2"
D10_ROOT = MYDRIVE / "ST-OMR-D10" / "stage7d10-authoritative-562c8fcfabf1b41573f1ef591d88ae65335ce16a"
for name, path in {"DATA_ROOT": DATA_ROOT, "CHECKPOINT_ROOT": CHECKPOINT_ROOT, "M4A_ROOT": M4A_ROOT, "D10_ROOT": D10_ROOT}.items():
    if not path.is_dir():
        raise RuntimeError(f"{name} bulunamadi: {path}")
print("DRIVE CHECK = PASS")

REPO_URL = f"https://github.com/{REPOSITORY}.git"
if not REPO.exists():
    subprocess.check_call(["git", "clone", "--no-checkout", REPO_URL, str(REPO)])
elif not (REPO / ".git").is_dir():
    raise RuntimeError(f"REPO git repository degil: {REPO}")
if "origin" not in subprocess.check_output(["git", "-C", str(REPO), "remote"], text=True).split():
    subprocess.check_call(["git", "-C", str(REPO), "remote", "add", "origin", REPO_URL])
else:
    subprocess.check_call(["git", "-C", str(REPO), "remote", "set-url", "origin", REPO_URL])
subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin", EXPECTED_HEAD, "--depth", "1"])
if subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "FETCH_HEAD"], text=True).strip() != EXPECTED_HEAD:
    raise RuntimeError("FETCH_HEAD mismatch")
subprocess.check_call(["git", "-C", str(REPO), "checkout", "--detach", EXPECTED_HEAD])
if subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip() != EXPECTED_HEAD:
    raise RuntimeError("HEAD mismatch")
if subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip():
    raise RuntimeError("Repository worktree temiz degil")
print("REPOSITORY CHECK = PASS")
print("HEAD =", EXPECTED_HEAD)
print("CI RUN ID =", EXPECTED_CI_RUN_ID)

ANN = DATA_ROOT / "annotations"
REPORT = ANN / "v5_3g_authoritative_rescue_training_report.json"
ARTIFACT_DIR = ANN / "v5_3g_authoritative_rescue_artifacts"
TEMP_ARTIFACT_DIR = ANN / ".v5_3g_authoritative_rescue_artifacts.tmp"
ENVELOPE = ANN / f"v5_3g_execution_envelope_{EXPECTED_HEAD}.json"
for path in (REPORT, ARTIFACT_DIR, TEMP_ARTIFACT_DIR, ENVELOPE):
    if path.exists():
        raise RuntimeError(f"Refusing overwrite/rerun: {path}")
print("OUTPUT GUARD = PASS")

# Colab global paketleri rescue runtime'a dahil edilmez.
if VENV.exists():
    shutil.rmtree(VENV)
subprocess.check_call([sys.executable, "-m", "venv", str(VENV)])
subprocess.check_call([str(VENV_PYTHON), "-m", "pip", "install", "-r", str(REPO / "requirements.txt")])
subprocess.check_call([
    str(VENV_PYTHON), "-m", "pip", "install",
    "--index-url", "https://download.pytorch.org/whl/cpu",
    "-r", str(REPO / "requirements-training.txt"),
])
subprocess.check_call([str(VENV_PYTHON), "-m", "pip", "check"])

EXPECTED_RUNTIME = {
    "lxml": "6.1.1", "verovio": "6.2.1", "CairoSVG": "2.8.2",
    "Pillow": "12.3.0", "scipy": "1.18.0", "torch": "2.13.0+cpu",
}
env = os.environ.copy()
env["PYTHONNOUSERSITE"] = "1"
env["V53H_EXPECTED_RUNTIME"] = json.dumps(EXPECTED_RUNTIME, sort_keys=True)
runtime_check = r'''
from importlib import metadata
import json, os, sys
if sys.prefix == sys.base_prefix:
    raise RuntimeError("not isolated")
expected = json.loads(os.environ["V53H_EXPECTED_RUNTIME"])
actual = {k: metadata.version(k) for k in expected}
bad = {k: (expected[k], actual[k]) for k in expected if expected[k] != actual[k]}
if bad:
    raise RuntimeError(f"Runtime mismatch: {bad}")
print(json.dumps(actual, sort_keys=True))
'''
print("ISOLATED RUNTIME =", subprocess.check_output([str(VENV_PYTHON), "-c", runtime_check], env=env, text=True).strip())
print("PINNED RUNTIME = PASS")

worker = r'''
from pathlib import Path
import os, subprocess, sys
HEAD = "b36a9d2f5daade2c3568cac8cbc736ca75ca435f"
REPO = Path("/content/st-omr-training")
MYDRIVE = Path("/content/drive/MyDrive")
DATA = MYDRIVE / "TEST" / "METER_V2_1500_PACKAGE_AB_CLEAN"
CHECKPOINT = MYDRIVE / "ST-OMR-METER-SPECIALISTS"
M4A = CHECKPOINT / "m4a-234-digit-specialist-dataset-freeze-v2"
D10 = MYDRIVE / "ST-OMR-D10" / "stage7d10-authoritative-562c8fcfabf1b41573f1ef591d88ae65335ce16a"
if sys.prefix == sys.base_prefix or os.environ.get("PYTHONNOUSERSITE") != "1":
    raise RuntimeError("isolated worker boundary missing")
if subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip() != HEAD:
    raise RuntimeError("worker HEAD mismatch")
sys.path.insert(0, str(REPO))
from st_omr_training import meter_v5_2b_specialist_adaptation as v52b
from st_omr_training import meter_v5_3e_rescue_training_preregistration_v1 as v53e
from st_omr_training import meter_v5_3g_authoritative_rescue_training_v1 as rescue
if rescue.V53F_HEAD_SHA != "7ed41f2872058ac5e3e52df756b9098a1d60052d":
    raise RuntimeError("V5-3F binding changed")
contract = rescue.execution_contract()
if contract["authoritative_group_counts"] != v53e.EXPECTED_TRAIN_GROUP_COUNTS:
    raise RuntimeError("group-count contract changed")
if contract.get("one_shot_non_overwriting") is not True or contract.get("exact_sha_colab_wrapper_required_for_external_execution") is not True:
    raise RuntimeError("execution boundary changed")
boundary = rescue.safety_boundary()
for key, expected in {
    "digit4_loaded": False, "digit4_frozen": True, "threshold_tuning": False,
    "hyperparameter_sweep": False, "automatic_second_configuration": False,
    "historical_validation_opened": False, "first30_opened": False,
    "v5_validation_opened": False, "final_holdout_locked": True,
    "resolver_wiring": False, "production_promotion": False,
}.items():
    if boundary.get(key) != expected:
        raise RuntimeError(f"safety boundary changed: {key}")
d2 = v52b.locate_checkpoint_by_sha_v1(CHECKPOINT, v52b.DIGIT2_SHA256)
d3 = v52b.locate_checkpoint_by_sha_v1(CHECKPOINT, v52b.DIGIT3_SHA256)
print("MODULE/CHECKPOINT/SAFETY = PASS")
rescue.run_authoritative_rescue_training_v1(
    DATA, m4a_root=M4A, d10_root=D10,
    digit2_frozen=d2, digit3_frozen=d3,
    confirmation=rescue.APPROVAL_TOKEN,
)
'''
subprocess.check_call([str(VENV_PYTHON), "-c", worker], env=env)

if not REPORT.is_file():
    raise RuntimeError("Authoritative report not written")
report_bytes = REPORT.read_bytes()
report = json.loads(report_bytes)
for key, expected in {
    "single_authoritative_execution_completed": True,
    "candidate_configuration_count": 1,
    "train_performance_gate_executed": False,
    "historical_validation_retention_executed": False,
    "first30_opened": False,
    "v5_validation_opened": False,
    "final_holdout_locked": True,
    "runtime_authority_changed": False,
    "production_promotion": False,
}.items():
    if report.get(key) != expected:
        raise RuntimeError(f"report boundary mismatch: {key}")
if report.get("numerical_integrity_gate", {}).get("gate") != "PASS":
    raise RuntimeError("numerical integrity failed")
if report.get("frozen_state_isolation_gate", {}).get("gate") != "PASS":
    raise RuntimeError("frozen-state isolation failed")

expected_counts = {
    "2": {"v5_frozen_false_negative_positive": 90, "v5_frozen_true_negative": 450, "historical_frozen_false_negative_positive": 14, "historical_frozen_true_negative": 25254},
    "3": {"v5_frozen_false_negative_positive": 90, "v5_frozen_true_negative": 450, "historical_frozen_false_negative_positive": 12, "historical_frozen_true_negative": 25364},
}
artifact_sha256, group_fingerprints = {}, {}
for digit in ("2", "3"):
    item = report["per_specialist"][digit]
    if item["materialization"]["group_counts"] != expected_counts[digit]:
        raise RuntimeError(f"{digit}-AI group counts changed")
    if item.get("frozen_state_bit_identical") is not True or item.get("frozen_state_before") != item.get("frozen_state_after"):
        raise RuntimeError(f"{digit}-AI frozen state changed")
    execution = item["execution"]
    if execution.get("authoritative_dataset_execution") is not True or execution.get("optimizer_steps") != 110:
        raise RuntimeError(f"{digit}-AI execution receipt invalid")
    if execution.get("checkpoint_write") is not False or execution.get("protected_evaluation_opened") is not False:
        raise RuntimeError(f"{digit}-AI protected boundary opened")
    artifact = item["artifact"]
    artifact_path = Path(artifact["artifact_path"])
    if not artifact_path.is_file() or artifact.get("reload_verified") is not True:
        raise RuntimeError(f"{digit}-AI rescue artifact invalid")
    actual_sha = hashlib.sha256(artifact_path.read_bytes()).hexdigest()
    if actual_sha != artifact["artifact_sha256"]:
        raise RuntimeError(f"{digit}-AI rescue artifact SHA mismatch")
    artifact_sha256[digit] = actual_sha
    group_fingerprints[digit] = item["materialization"]["group_fingerprints"]

if subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip() != EXPECTED_HEAD:
    raise RuntimeError("Post-run HEAD mismatch")
if subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip():
    raise RuntimeError("Post-run worktree dirty")

envelope = {
    "schema": "st-omr-meter-v5-3h-authoritative-rescue-execution-envelope-v1",
    "repository": REPOSITORY, "expected_head": EXPECTED_HEAD, "actual_head": EXPECTED_HEAD,
    "ci_run_id": EXPECTED_CI_RUN_ID, "executed_at_utc": datetime.now(timezone.utc).isoformat(),
    "report_sha256": hashlib.sha256(report_bytes).hexdigest(),
    "artifact_sha256": artifact_sha256, "group_fingerprints": group_fingerprints,
    "single_authoritative_execution_completed": True, "candidate_configuration_count": 1,
    "numerical_integrity_gate": "PASS", "frozen_state_isolation_gate": "PASS",
    "train_performance_gate_executed": False, "historical_validation_opened": False,
    "first30_opened": False, "v5_reserve_opened": False, "v5_validation_opened": False,
    "final_holdout_locked": True, "digit4_frozen": True, "threshold_tuning": False,
    "hyperparameter_sweep": False, "automatic_second_configuration": False,
    "runtime_authority_changed": False, "production_promotion": False,
    "isolated_runtime": True, "python_no_user_site": True,
}
tmp = ENVELOPE.with_suffix(ENVELOPE.suffix + ".tmp")
tmp.write_text(json.dumps(envelope, indent=2, sort_keys=True) + "\n", encoding="utf-8")
os.replace(tmp, ENVELOPE)
print("V5-3G AUTHORITATIVE TRAIN EXECUTION = PASS")
print("NUMERICAL INTEGRITY = PASS | FROZEN STATE ISOLATION = PASS")
print("TRAIN PERFORMANCE/HISTORICAL VALIDATION/FIRST-30/V5 VAL = CLOSED")
print("FINAL_HOLDOUT = LOCKED")
print("REPORT =", REPORT)
print("ENVELOPE =", ENVELOPE)
